# Environment Setup

In [ ]:
%%capture
!pip install --upgrade pip
!pip install jiwer evaluate tensorboard datasets librosa soundfile
!pip install --upgrade transformers torch torchvision torchaudio accelerate
!pip install numpy==2.1.0 scipy==1.11.4 librosa==0.10.1 numba==0.58.1
!pip install typing_extensions --upgrade

In [ ]:
from huggingface_hub import login
login(token="HUGGING_FACE_TOKEN")

# Data Loading Pipeline

In [ ]:
from datasets import load_dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Telugu", task="transcribe")

ds = load_dataset("kaarthu2003/SlrCvVoicesTtsDataset", streaming=True)

train_data = ds["train"]
test_data = ds["validation"]

print("Streaming pipeline ready.")

# Preprocessing and Text Cleaning

In [ ]:
import re
import librosa

telugu_special_unwanted_characters = ['ౄ','ౢ','ౣ','ౠ','ఽ','౦','౧','౨','౩','౪','౫','౬','౭','౮','౯','ఀ','ౘ','ౙ','ౚ','౷','‘','’','“','”','%','.',';','-',',','/','\\','_','&','G','P','S','e','l','n','r','t','\u200c']
chars_to_remove_regex = f'[{re.escape("".join(telugu_special_unwanted_characters))}]'

def prepare_dataset(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"])
    audio = batch["audio"]
    if audio["sampling_rate"] != 16000:
        audio["array"] = librosa.resample(audio["array"], orig_sr=audio["sampling_rate"], target_sr=16000)
    
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=16000).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

train_data = train_data.map(prepare_dataset)
test_data = test_data.map(prepare_dataset)

# Model Initialization

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Telugu", task="transcribe")

model.generation_config.language = "telugu"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

# Section 6: Trainer Configuration and Fine-Tuning Setup

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="whisper-train-telugu",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=19760,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()

# Final Performance

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

model.eval()
predictions, references = [], []

with torch.no_grad():
    for example in test_data.take(100): # Sample evaluation
        input_features = torch.tensor(example["input_features"]).unsqueeze(0).to("cuda")
        predicted_ids = model.generate(input_features)
        predictions.append(processor.batch_decode(predicted_ids, skip_special_tokens=True)[0])
        references.append(processor.batch_decode([example["labels"]], skip_special_tokens=True)[0])

print(f"Final WER: {100 * wer_metric.compute(predictions=predictions, references=references):.2f}%")
print(f"Final CER: {100 * cer_metric.compute(predictions=predictions, references=references):.2f}%")